# SGD vs Adam — Visual Comparison

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

SGD takes the same step in every direction; Adam adapts the step per-parameter using running averages of the gradient and its square. Adam usually converges faster but can generalise slightly worse than well-tuned SGD with momentum.


## Mathematical Formulation

SGD with momentum:
$$v_t = \mu v_{t-1} + g_t,\quad \theta_{t+1} = \theta_t - \eta v_t$$

Adam:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\quad v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\theta_{t+1} = \theta_t - \eta\,\hat m_t / (\sqrt{\hat v_t} + \varepsilon)$$


## Implementation


In [ ]:
import torch
import matplotlib.pyplot as plt


In [ ]:
def himmelblau(x, y):  # banana-shaped loss landscape
    return (x*x + y - 11)**2 + (x + y*y - 7)**2

def run(opt_cls, lr, steps=200, start=(-3.0, 2.0)):
    p = torch.tensor(start, requires_grad=True)
    opt = opt_cls([p], lr=lr) if opt_cls is torch.optim.SGD else opt_cls([p], lr=lr)
    traj = [p.detach().clone()]
    for _ in range(steps):
        loss = himmelblau(p[0], p[1])
        opt.zero_grad(); loss.backward(); opt.step()
        traj.append(p.detach().clone())
    return torch.stack(traj)


## Experiment


In [ ]:
sgd_traj = run(torch.optim.SGD, lr=0.005)
adam_traj = run(torch.optim.Adam, lr=0.1)

xs = torch.linspace(-5, 5, 200)
ys = torch.linspace(-5, 5, 200)
X, Y = torch.meshgrid(xs, ys, indexing='xy')
Z = himmelblau(X, Y)

plt.contourf(X, Y, Z.numpy(), levels=30, cmap='Greys')
plt.plot(sgd_traj[:, 0], sgd_traj[:, 1], label='SGD', lw=2)
plt.plot(adam_traj[:, 0], adam_traj[:, 1], label='Adam', lw=2)
plt.legend(); plt.title('Trajectories on Himmelblau'); plt.show()


## Discussion

- Adam's per-parameter adaptive step is what lets it handle highly anisotropic loss surfaces.
- Tune Adam with `weight_decay` (use **AdamW**) — otherwise weight decay couples with adaptive step and behaves unexpectedly.
- For computer vision, SGD + momentum + LR schedule still wins on ImageNet-scale; Adam tends to win on NLP.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
